# 5.02 Multi-Vector · Parent-Document Retriever

원본 문서 하나당 **여러 임베딩**을 생성해 검색 품질을 올리는 두 retriever를 다룬다.
- `MultiVectorRetriever` — 임의의 파생 텍스트(요약 · 가설 질문 · 청크)를 임베딩해서 vector DB에 넣고, 원본 문서는 별도 `docstore`에 둔다
- `ParentDocumentRetriever` — 작은 청크로 **검색**하지만 최종 반환은 **부모 문단(원본)**. MultiVectorRetriever의 특수 케이스

## 학습 목표

- `MultiVectorRetriever(vectorstore=..., docstore=..., id_key=...)`의 3층 구조를 이해한다
- 세 전략 (요약 / 가설 질문 / 청크) 중 어느 것이 어떤 상황에 맞는지 안다
- `ParentDocumentRetriever(child_splitter=..., parent_splitter=...)`로 "작게 검색 · 크게 반환" 패턴 구현
- `InMemoryByteStore` 를 docstore로 써서 예제를 메모리에서 완결한다

## 언제 쓰나

- **긴 문서** (10페이지+ 계약서·매뉴얼): 작은 청크로 정밀 검색, LLM에는 주변 문맥까지 풍부하게 넘겨야 할 때
- 각 원본 문서를 **관점 여러 개**로 검색시키고 싶을 때 (한 줄 요약으로도 걸리고, 가설 질문으로도 걸리게)
- RAG 답변 품질이 **청크 크기**에 과민하게 반응할 때 — parent-document 패턴이 흔한 해결책


## 5.02.1 환경 설정

`MultiVectorRetriever` / `ParentDocumentRetriever`는 `langchain-classic`에 있다. 예제에서는 요약 생성을 OpenAI로 하므로 `OPENAI_API_KEY` 필요.


In [ ]:
# !pip install -U langchain langchain-classic langchain-core langchain-openai langchain-text-splitters

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요 (embed + 요약 LLM)"


## 5.02.2 기본 사용법 — MultiVectorRetriever (청크 전략)

가장 단순한 variant: **원본 문서를 잘게 쪼개 청크를 임베딩**한다. Retriever는 청크로 검색한 후 `docstore`에서 원본을 꺼낸다.

3층 구조:
1. `vectorstore` — 파생(청크/요약/질문)의 임베딩
2. `docstore` (key-value) — 원본 문서 (`InMemoryByteStore` 면 메모리)
3. `id_key` — 파생 문서 metadata에 넣는 **부모 ID** 필드명 (기본 `"doc_id"`)


In [ ]:
import uuid
from langchain_core.documents import Document
from langchain_core.stores import InMemoryByteStore
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import MultiVectorRetriever

# 긴 원본 문서 3개
docs = [
    Document(
        page_content=(
            "LangChain은 LLM 애플리케이션을 구축하기 위한 오픈소스 프레임워크이다. "
            "표준 인터페이스(chat model, embedding, vectorstore, retriever, tool)를 제공하고, "
            "v1에서는 create_agent · middleware · streaming 모델을 중심으로 API가 정리되었다. "
            "Python과 TypeScript 두 언어로 구현체가 제공되며, LangGraph · LangSmith와 결합해 "
            "에이전트 오케스트레이션과 관측을 완성하는 것이 표준 스택이다."
        ),
        metadata={"source": "overview"},
    ),
    Document(
        page_content=(
            "LangGraph는 상태 기반 그래프로 멀티 에이전트 워크플로우를 선언하는 라이브러리이다. "
            "노드가 상태를 업데이트하고 엣지가 흐름을 결정한다. checkpointer로 세션 영속화, "
            "Store로 장기 메모리를 관리한다. interrupt()와 Command(resume=...)로 human-in-the-loop을 넣는다. "
            "LangChain 위에 얹혀 돌 수도 있고 단독으로도 쓸 수 있다."
        ),
        metadata={"source": "langgraph"},
    ),
    Document(
        page_content=(
            "RAG(Retrieval-Augmented Generation)는 외부 지식을 LLM 응답에 결합하는 기법이다. "
            "가장 전형적인 구성은 문서 로더 → 스플리터 → 임베딩 → 벡터 스토어 → retriever → 모델로 이어진다. "
            "품질 문제 대부분은 retriever 단계에서 발생하며, multi-vector · parent-document · hybrid ensemble 같은 "
            "기법이 점수를 크게 올린다."
        ),
        metadata={"source": "rag-intro"},
    ),
]

# 부모 ID 부여 & 청크 생성
id_key = "doc_id"
doc_ids = [str(uuid.uuid4()) for _ in docs]
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)

sub_docs = []
for i, d in enumerate(docs):
    for chunk in splitter.split_documents([d]):
        chunk.metadata[id_key] = doc_ids[i]
        sub_docs.append(chunk)

print("원본:", len(docs), "· 청크:", len(sub_docs))

vectorstore = InMemoryVectorStore(embedding=OpenAIEmbeddings(model="text-embedding-3-small"))
vectorstore.add_documents(sub_docs)

byte_store = InMemoryByteStore()

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=byte_store,   # key-value 원본 저장소
    id_key=id_key,
    search_kwargs={"k": 4},  # 벡터 검색 단계 k
)
# 부모 문서를 docstore에 올려둔다
retriever.docstore.mset(list(zip(doc_ids, docs)))

hits = retriever.invoke("에이전트 워크플로우")
for d in hits:
    print("-", d.metadata.get("source"), "|", d.page_content[:60])


## 5.02.3 파생 전략 — 요약 · 가설 질문 · 청크

같은 Multi-Vector 구조에서 파생 텍스트만 바꾸면 세 가지 전략이 된다.

| 전략 | 파생 임베딩 대상 | 강점 |
|------|------------------|------|
| **청크**(위 예제) | 원본을 잘게 쪼갠 조각 | 구현 가장 단순, 키워드 일치 강함 |
| **요약** | LLM이 만든 1~3문장 한줄요약 | 긴 문서에서 핵심만 임베딩해 의미 매칭이 깔끔해짐 |
| **가설 질문** | LLM이 "이 문서는 어떤 질문에 답하는가?" 로 만든 3~5개 Q | 사용자 질문과 embedding 공간 정렬이 좋아 검색 품질 상승 |

아래는 **요약 전략** 최소 구현. LLM 호출 수를 줄이려고 문서 3개만 사용한다.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "아래 문서를 한 문장(40자 이하)으로 요약한다. 설명 없이 요약만 출력."),
    ("human", "{doc}"),
])
summary_chain = summary_prompt | llm

summaries = [summary_chain.invoke({"doc": d.page_content}).content for d in docs]
for s in summaries:
    print("요약:", s)

# 새 vectorstore에 요약만 임베딩
summary_store = InMemoryVectorStore(embedding=OpenAIEmbeddings(model="text-embedding-3-small"))
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]}) for i, s in enumerate(summaries)
]
summary_store.add_documents(summary_docs)

retriever_summary = MultiVectorRetriever(
    vectorstore=summary_store,
    byte_store=InMemoryByteStore(),
    id_key=id_key,
    search_kwargs={"k": 2},
)
retriever_summary.docstore.mset(list(zip(doc_ids, docs)))

for d in retriever_summary.invoke("체크포인트와 장기 메모리"):
    print("-", d.metadata.get("source"), "|", d.page_content[:60])


## 5.02.4 결과 비교 — 청크 전략 vs 요약 전략

같은 질의 `"human-in-the-loop"`을 두 retriever에 돌린다.
- **청크 전략**은 원본 속 문구 `interrupt()` 같은 **표면 어휘**에 민감
- **요약 전략**은 원본 문구가 요약에 포함됐는지에 좌우 — 놓치면 검색 실패, 반대로 요약 품질이 좋으면 매우 강함


In [ ]:
query = "human-in-the-loop"

print("[청크 전략]")
for d in retriever.invoke(query):
    print("-", d.metadata.get("source"))

print("\n[요약 전략]")
for d in retriever_summary.invoke(query):
    print("-", d.metadata.get("source"))


## 5.02.5 ParentDocumentRetriever — 작게 검색, 크게 반환

`ParentDocumentRetriever`는 MultiVector의 **특수 케이스**로, 스플리터를 두 단계로 주면 자동으로:
1. `parent_splitter`로 원본을 큰 덩어리로 쪼갬 (docstore에 저장)
2. `child_splitter`로 **그 안에서 다시** 작게 쪼갬 (vectorstore에 임베딩)
3. child hit → 부모 덩어리를 LLM에 반환

`parent_splitter`를 None으로 두면 원본 전체가 부모 단위가 된다.


In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=80,  chunk_overlap=10)

pdr_store = InMemoryVectorStore(embedding=OpenAIEmbeddings(model="text-embedding-3-small"))
pdr = ParentDocumentRetriever(
    vectorstore=pdr_store,
    docstore=InMemoryByteStore(),  # parent 저장소
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 2},
)

pdr.add_documents(docs, ids=None)

results = pdr.invoke("checkpoint 세션 영속화")
print("반환 문서 수:", len(results))
for d in results:
    print("- len =", len(d.page_content), "| preview:", d.page_content[:80])


## 5.02.6 `create_agent` 연동

`MultiVectorRetriever` / `ParentDocumentRetriever` 모두 `BaseRetriever`이므로 `create_retriever_tool`로 그대로 도구화된다. 에이전트는 tool이 무슨 전략을 쓰는지 알 필요 없다 — 인터페이스는 `"query: str" → "formatted docs: str"`.


In [ ]:
from langchain_classic.tools.retriever import create_retriever_tool
from langchain.agents import create_agent

mv_tool = create_retriever_tool(
    retriever,
    name="search_docs_chunks",
    description="LangChain/LangGraph/RAG 관련 내부 문서를 multi-vector 청크 전략으로 검색한다.",
)
pdr_tool = create_retriever_tool(
    pdr,
    name="search_docs_parent",
    description="같은 문서 코퍼스에서 작은 청크로 찾고 부모 문단을 반환한다 — 문맥 보존용.",
)

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[mv_tool, pdr_tool],
    system_prompt="사용자 질문에 답하기 전에 최소 한 번은 두 도구 중 하나를 호출하라.",
)

r = agent.invoke({"messages": [{"role": "user", "content": "LangGraph에서 세션 상태는 어떻게 보관하지?"}]})
print(r["messages"][-1].content[:500])


## 체크리스트

- [ ] `MultiVectorRetriever`의 `id_key` 값은 파생 문서 metadata와 docstore key가 **반드시** 일치해야 함
- [ ] 요약·가설 질문 전략은 LLM 호출 비용 — 인덱싱 한 번만 돌리고 결과를 캐시 (`byte_store` 지속화 스토리지 사용)
- [ ] `ParentDocumentRetriever.add_documents()`가 자동으로 split + docstore 주입까지 해 준다
- [ ] `docstore`가 메모리(`InMemoryByteStore`)면 프로세스 재시작 시 원본 유실 — 프로덕션은 Redis/FS/SQLite 등 영속 backend로 교체

## 다음

- `03_self_query.ipynb` — 자연어를 metadata filter로 자동 변환
- `05_advanced/05_agentic_rag.ipynb` — multi-vector + agent 합성

## 참고

- 공식 가이드: https://python.langchain.com/docs/how_to/multi_vector/
- parent-document 패턴: https://python.langchain.com/docs/how_to/parent_document_retriever/
